# E/22/194

# Assignment 7b: Gaussian Mixture Model Clustering as Conditional Updating

## Q. Bayesian Estimation of a User Ability Parameter from Item Responses

### 1. Visualizing the 2PL response curves

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.special import expit

In [ ]:
# Range of possible ability values
theta = np.linspace(-4, 4, 1000)

In [ ]:
# Two discrimination values are used.
# For a = 2.0, three different difficulty values are used.
item_settings = [
    (0.5, 0.0),
    (1.5, -1.0),
    (1.5, 0.0),
    (1.5, 1.0)
]

fig = go.Figure()

for a, b in item_settings:
    probability = expit(a * (theta - b))

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=probability,
            mode="lines",
            name=f"a = {a}, b = {b}"
        )
    )

fig.update_layout(
    title="2PL Probability of a Correct Response",
    xaxis_title="Ability, θ",
    yaxis_title="P(Yᵢ = 1 | Θ = θ)",
    template="plotly_white",
    width=900,
    height=550
)

fig.update_yaxes(range=[0, 1])

fig.show()



**Increasing $b_i$** shifts the curve to the right. This means that a higher ability is required to obtain the same probability of success. **Decreasing $b_i$** shifts the curve to the left, indicating an easier item.

The discrimination parameter $a_i$ controls the steepness of the curve. A **large $a_i$** produces a steep curve and strongly separates users with ability below and above $b_i$. A **small $a_i$** produces a flatter curve and provides less distinction between ability levels.

### 2. Sequential Likelihood Contribution

For the new response $y_k$, where $y_k \in \{0,1\}$, the single-response likelihood is:

$$L(y_k \mid \theta) = p_k(\theta)^{y_k} \left[1 - p_k(\theta)\right]^{1-y_k}$$

If the response is correct, $y_k = 1$, then:

$$L(1 \mid \theta) = p_k(\theta)$$

If the response is incorrect, $y_k = 0$, then:

$$L(0 \mid \theta) = 1 - p_k(\theta)$$

Assuming that item responses are conditionally independent given the user's ability, the joint likelihood for the running response history $\mathbf{y}^{(k)} = (y_1, y_2, \ldots, y_k)$ is:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} p_i(\theta)^{y_i} \left[1 - p_i(\theta)\right]^{1-y_i}$$



### 3. Mathematical Formulation of the Running Update

At step $k-1$, the current posterior density is:

$$f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)})$$

When the new response $y_k$ is observed:

$$f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)}) \propto L(y_k\mid\theta) f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)})$$

Using the Bernoulli likelihood:

$$f_k(\theta) \propto p_k(\theta)^{y_k} \left[1-p_k(\theta)\right]^{1-y_k} f_{k-1}(\theta)$$

The normalized posterior is:

$$f_k(\theta) = \frac{L(y_k\mid\theta)f_{k-1}(\theta)}{\displaystyle \int_{-\infty}^{\infty} L(y_k\mid s)f_{k-1}(s)\,ds}$$



### 4. Dynamic Shifting

For low ability values, particularly when $\theta \ll b_k$, the probability $p_k(\theta)$ is very small. Therefore, the previous posterior density at low ability values is multiplied by a small number.

For ability values close to or greater than $b_k$, the probability of a correct response is larger. These ability values retain more posterior probability after the update.


$$\log f_k(\theta) = \log f_{k-1}(\theta) + \log p_k(\theta) + C$$

where $C$ is a normalization constant.

For a correct response,

$$\frac{d}{d\theta}\log p_k(\theta) = a_k[1-p_k(\theta)] > 0$$

Therefore, the new likelihood adds positive weight to larger values of $\theta$. As a result, the posterior peak, posterior mean, and MAP estimate normally shift toward a higher ability value.

A correct response to a highly difficult item is therefore strong evidence that the user may have high ability.

### 5. Tracking Certainty and Sharpness

The discrimination parameter $a_k$ controls the steepness of the item response curve.

The information supplied by item $k$ is:

$$I_k(\theta) = a_k^2 p_k(\theta)[1-p_k(\theta)]$$

This information is largest when $p_k(\theta) = 0.5$, which occurs near $\theta = b_k$.

**When $a_k$ is large:**

A large $a_k$ gives a steep response curve. Near the item difficulty $b_k$, small differences in ability produce large differences in the probability of a correct response. Consequently, the response strongly separates ability values below and above $b_k$. The running posterior may become narrower, and its variance may decrease considerably.

**When $a_k$ is small:**

A small $a_k$ gives a flatter response curve. The probability of a correct response changes only slightly over a broad range of ability values. Therefore, the response gives less information about ability. The posterior changes less, and the reduction in posterior variance is smaller.



### 6. Numerical Implementation of a Running Grid

For example let $-4 \leq \theta \leq 4$.

The algorithm is

1. **Evaluate the initial standard normal prior at every grid point:**
   $$f_0(\theta_j) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta_j^2}{2}\right)$$

2. **For item $k$, calculate the correct-response probability at every grid point:**
   $$p_k(\theta_j) = \frac{1}{1+\exp[-a_k(\theta_j-b_k)]}$$

3. **Calculate the likelihood contribution:**
   $$L_j = p_k(\theta_j)^{y_k} [1-p_k(\theta_j)]^{1-y_k}$$

4. **Multiply the likelihood by the previous posterior:**
   $$q_j = L_j f_{k-1}(\theta_j)$$

5. **Calculate the numerical normalizing constant:**
   $$Z_k \approx \int q(\theta)\,d\theta$$

6. **Normalize the posterior:**
   $$f_k(\theta_j) = \frac{q_j}{Z_k}$$


The posterior mean is calculated using:

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} \approx \int \theta f_k(\theta)\,d\theta$$

while the MAP estimate is obtained from the grid point with the largest posterior value:

$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} \approx \theta_{\operatorname{argmax}_j f_k(\theta_j)}$$

This process is repeated after every new response.

### 7. Evaluating Convergence over the Timeline

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go


In [ ]:
from scipy.special import expit
from scipy.integrate import trapezoid

In [ ]:

# ---------------------------------------------------------
# Simulation settings
# ---------------------------------------------------------
rng = np.random.default_rng(42)

theta_true = 0.75
n_items = 20

# Fixed grid for the ability parameter
theta_grid = np.linspace(-4, 4, 4001)


In [ ]:
# ---------------------------------------------------------
# Initial standard normal prior
# ---------------------------------------------------------
posterior = (
    np.exp(-0.5 * theta_grid**2)
    / np.sqrt(2 * np.pi)
)

# Normalize the prior numerically
posterior = posterior / trapezoid(
    posterior,
    theta_grid
)


In [ ]:
# ---------------------------------------------------------
# Step 0 estimates
# ---------------------------------------------------------
initial_mean = trapezoid(
    theta_grid * posterior,
    theta_grid
)

initial_map = theta_grid[
    np.argmax(posterior)
]

initial_variance = trapezoid(
    (theta_grid - initial_mean)**2 * posterior,
    theta_grid
)

steps = [0]
posterior_means = [initial_mean]
map_estimates = [initial_map]
posterior_variances = [initial_variance]

records = []

In [ ]:
# ---------------------------------------------------------
# Sequential Bayesian updating
# ---------------------------------------------------------
for k in range(1, n_items + 1):

    # Generate random item parameters
    a_k = rng.uniform(0.5, 2.0)
    b_k = rng.normal(0.0, 1.0)

    # Probability of a correct response for the true ability
    p_true = expit(
        a_k * (theta_true - b_k)
    )

    # Simulate the user's response
    uniform_draw = rng.uniform(0.0, 1.0)
    y_k = int(uniform_draw < p_true)

    # Correct-response probabilities across the grid
    p_grid = expit(
        a_k * (theta_grid - b_k)
    )

    # Likelihood contribution of the new response
    likelihood = (
        p_grid**y_k
        * (1.0 - p_grid)**(1 - y_k)
    )

    # Unnormalized running posterior
    unnormalized_posterior = (
        likelihood * posterior
    )

    # Sequential normalization
    normalizing_constant = trapezoid(
        unnormalized_posterior,
        theta_grid
    )

    if (
        not np.isfinite(normalizing_constant)
        or normalizing_constant <= 0
    ):
        raise RuntimeError(
            f"Invalid normalizing constant at step {k}"
        )

    posterior = (
        unnormalized_posterior
        / normalizing_constant
    )

    # Running posterior mean
    posterior_mean = trapezoid(
        theta_grid * posterior,
        theta_grid
    )

    # Running MAP estimate
    posterior_map = theta_grid[
        np.argmax(posterior)
    ]

    # Running posterior variance
    posterior_variance = trapezoid(
        (theta_grid - posterior_mean)**2
        * posterior,
        theta_grid
    )

    # Store estimates
    steps.append(k)
    posterior_means.append(posterior_mean)
    map_estimates.append(posterior_map)
    posterior_variances.append(
        posterior_variance
    )

    records.append({
        "Step": k,
        "Discrimination a": a_k,
        "Difficulty b": b_k,
        "True correct probability": p_true,
        "Response y": y_k,
        "Posterior mean": posterior_mean,
        "MAP estimate": posterior_map,
        "Posterior variance": posterior_variance
    })


In [ ]:
# ---------------------------------------------------------
# Display the running results
# ---------------------------------------------------------
results = pd.DataFrame(records)

display(results.round(4))

print(
    f"Final posterior mean: "
    f"{posterior_means[-1]:.4f}"
)

print(
    f"Final MAP estimate: "
    f"{map_estimates[-1]:.4f}"
)

print(
    f"Final posterior variance: "
    f"{posterior_variances[-1]:.4f}"
)

print(
    f"Final posterior standard deviation: "
    f"{np.sqrt(posterior_variances[-1]):.4f}"
)

,Step,Discrimination a,Difficulty b,True correct probability,Response y,Posterior mean,MAP estimate,Posterior variance
0,1,1.6609,-1.0400,0.9513,1,0.2760,0.190,0.7769
1,2,1.5461,-1.9510,0.9849,1,0.3364,0.230,0.7306
2,3,1.6417,-0.3162,0.8520,1,0.6112,0.496,0.6122
3,4,1.1756,0.8794,0.4620,0,0.3459,0.278,0.4814
4,5,1.4658,1.1272,0.3652,0,0.1770,0.150,0.3953
5,6,0.8409,0.3688,0.5795,1,0.3449,0.318,0.3789
6,7,1.7414,-0.0499,0.8011,1,0.5396,0.500,0.3237
7,8,1.0318,1.2225,0.3805,0,0.4324,0.404,0.2943
8,9,1.6676,-0.3521,0.8627,1,0.5361,0.498,0.2671
9,10,0.5657,0.4127,0.5476,0,0.4597,0.426,0.2561


Final posterior mean: 0.6551
Final MAP estimate: 0.6300
Final posterior variance: 0.1372
Final posterior standard deviation: 0.3705


In [ ]:
# ---------------------------------------------------------
# Plot both running estimators
# ---------------------------------------------------------
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True ability = 0.75",
    annotation_position="top left"
)

fig.update_layout(
    title="Sequential Bayesian Estimation of User Ability",
    xaxis_title="Number of Observed Items, k",
    yaxis_title="Estimated Ability",
    template="plotly_white",
    width=950,
    height=550
)

fig.update_xaxes(
    tickmode="linear",
    dtick=1
)

fig.show()

At step 0, both estimators begin close to zero because the prior distribution is:

$$\Theta \sim \mathcal{N}(0,1)$$

The true hidden ability is:

$$\theta_{\text{true}} = 0.75$$

As each response is observed, the previous posterior is multiplied by the likelihood contribution of the new answer and normalized. Correct answers to informative or difficult items generally move the estimate upward, while incorrect answers to informative or easy items generally move it downward.

The paths of the posterior mean and MAP estimate do not necessarily approach 0.75 smoothly. Since the responses are randomly generated, an unexpected response can temporarily move an estimator farther from the true value.

Using the fixed random seed 42, the final estimates after 20 items are approximately:

$$\widehat{\theta}_{\mathrm{Bayes}}^{(20)} \approx 0.6551$$

and

$$\widehat{\theta}_{\mathrm{MAP}}^{(20)} \approx 0.6300$$

The final estimates are reasonably close to the true value $\theta_{\text{true}} = 0.75$.

The posterior variance decreases from approximately 1.00 at step 0 to approximately 0.1372 after step 20.

This decrease shows that the posterior distribution becomes narrower as more responses are observed. Therefore, the platform becomes more confident in its estimate of the user's ability.

However, confidence should not be evaluated only using the distance between the point estimate and the true value. Confidence is represented more directly by the posterior variance. A smaller posterior variance means that the platform considers a narrower range of ability values plausible.



## Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

### 1. Structural Probability and Properties

For $\Theta \sim \operatorname{Beta}(\alpha,\beta)$, the PDF is:

$$f(\theta\mid\alpha,\beta) = \frac{1}{B(\alpha,\beta)} \theta^{\alpha-1}(1-\theta)^{\beta-1}, \qquad 0\leq\theta\leq1$$

and

$$\mathbb{E}[\Theta] = \frac{\alpha}{\alpha+\beta}$$

For the given distributions:

* $\operatorname{Beta}(1,1)$: $\mathbb{E}[\Theta] = \frac{1}{2} = 0.5$
* $\operatorname{Beta}(2,8)$: $\mathbb{E}[\Theta] = \frac{2}{10} = 0.2$
* $\operatorname{Beta}(8,2)$: $\mathbb{E}[\Theta] = \frac{8}{10} = 0.8$

Therefore,

$$
\begin{aligned}
\alpha=\beta &\Rightarrow \text{density centered near } 0.5, \\
\alpha<\beta &\Rightarrow \text{mass shifts toward } 0, \\
\alpha>\beta &\Rightarrow \text{mass shifts toward } 1.
\end{aligned}
$$

In [3]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta = np.linspace(0.001, 0.999, 1000)

parameter_pairs = [
    (1, 1, "Beta(1,1): Uniform"),
    (2, 8, "Beta(2,8): Right-skewed"),
    (8, 2, "Beta(8,2): Left-skewed")
]

fig = go.Figure()

for alpha, beta_param, label in parameter_pairs:
    density = beta.pdf(theta, alpha, beta_param)

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=density,
            mode="lines",
            name=label
        )
    )

fig.update_layout(
    title="Beta Probability Density Functions",
    xaxis_title="Click-through rate, θ",
    yaxis_title="Probability density",
    template="plotly_white",
    width=900,
    height=520
)

fig.show()

### 2.Sequential Likelihood and Joint History

For one Bernoulli response,

$$Y_k \mid \Theta=\theta \sim \operatorname{Bernoulli}(\theta)$$

Therefore,

$$L(y_k \mid \theta) = P(Y_k=y_k \mid \Theta=\theta) = \theta^{y_k}(1-\theta)^{1-y_k}$$

because $y_k \in \{0,1\}$.

For $y_k = 1$,

$$L(1 \mid \theta) = \theta$$

For $y_k = 0$,

$$L(0 \mid \theta) = 1 - \theta$$

Assuming conditional independence,

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} \theta^{y_i}(1-\theta)^{1-y_i}$$

Let

$$S_k = \sum_{i=1}^{k} y_i$$

be the number of clicks. Then the number of non-clicks is $k - S_k$.

Hence,

$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{S_k}(1-\theta)^{k-S_k}$$

### 3. Closed-Form Analytical Updates (Conjugacy)

For one Bernoulli response,

$$Y_k \mid \Theta=\theta \sim \operatorname{Bernoulli}(\theta)$$

Therefore,

$$L(y_k \mid \theta) = P(Y_k=y_k \mid \Theta=\theta) = \theta^{y_k}(1-\theta)^{1-y_k}$$

because $y_k \in \{0,1\}$.

For $y_k = 1$,

$$L(1 \mid \theta) = \theta$$

For $y_k = 0$,

$$L(0 \mid \theta) = 1 - \theta$$

Assuming conditional independence,

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} \theta^{y_i}(1-\theta)^{1-y_i}$$

Let

$$S_k = \sum_{i=1}^{k} y_i$$

be the number of clicks. Then the number of non-clicks is $k - S_k$.

Hence,

$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{S_k}(1-\theta)^{k-S_k}$$

### 4. Dynamic Shifting Mechanics

**Click:** $y_k=1$

$$\alpha_k = \alpha_{k-1} + 1, \qquad \beta_k = \beta_{k-1}$$

Therefore,

$$f_k(\theta) \propto \theta f_{k-1}(\theta)$$

Large values of $\theta$ receive greater weight. Hence,

$$y_k=1 \Rightarrow \text{posterior shifts toward larger CTR values}$$

**Non-click:** $y_k=0$

$$\alpha_k = \alpha_{k-1}, \qquad \beta_k = \beta_{k-1} + 1$$

Therefore,

$$f_k(\theta) \propto (1-\theta)f_{k-1}(\theta)$$

Small values of $\theta$ receive greater relative weight. Hence,

$$y_k=0 \Rightarrow \text{posterior shifts toward smaller CTR values}$$

For $\alpha,\beta>1$, the Beta mode is:

$$\operatorname{Mode}(\Theta) = \frac{\alpha-1}{\alpha+\beta-2}$$

After a click:

$$\operatorname{Mode}_k = \frac{\alpha_{k-1}}{\alpha_{k-1}+\beta_{k-1}-1}$$

After a non-click:

$$\operatorname{Mode}_k = \frac{\alpha_{k-1}-1}{\alpha_{k-1}+\beta_{k-1}-1}$$

### Comparison with a non-conjugate model

**Beta–Bernoulli:**

$$(\alpha_{k-1},\beta_{k-1}) \rightarrow (\alpha_k,\beta_k)$$

using simple arithmetic.

For a non-conjugate model such as 2PL IRT,

$$f_k(\theta) \propto L(y_k\mid\theta)f_{k-1}(\theta)$$

but the product is not a known standard distribution. Therefore,

$$\text{2PL posterior} \Rightarrow \text{grid integration, MCMC, or another numerical method}$$

### 5. Running Point Estimators

### Posterior mean

$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k+\beta_k} $$

Using the total number of clicks $S_k$,

$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_0+S_k}{\alpha_0+\beta_0+k} $$

### MAP estimate

For $\alpha_k>1$ and $\beta_k>1$,

$$ \widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k-1}{\alpha_k+\beta_k-2} $$

**Boundary cases:**

$$
\widehat{\theta}_{\mathrm{MAP}}^{(k)} =
\begin{cases}
\dfrac{\alpha_k-1}{\alpha_k+\beta_k-2}, &\alpha_k>1,\ \beta_k>1,\\[8pt]
0, &\alpha_k\leq1,\ \beta_k>1,\\[4pt]
1, &\alpha_k>1,\ \beta_k\leq1.
\end{cases}
$$

For $\alpha_k=\beta_k=1$, the density is uniform, so every point in $[0,1]$ is a mode. For plotting, we may use:

$$ \widehat{\theta}_{\mathrm{MAP}}^{(0)} = 0.5 $$

### Posterior variance

The posterior variance is:

$$ \operatorname{Var}(\Theta\mid\mathbf{y}^{(k)}) = \frac{\alpha_k\beta_k}{(\alpha_k+\beta_k)^2(\alpha_k+\beta_k+1)} $$

### 5. Running Point Estimators

### Posterior mean

$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k+\beta_k} $$

Using the total number of clicks $S_k$,

$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_0+S_k}{\alpha_0+\beta_0+k} $$

### MAP estimate

For $\alpha_k>1$ and $\beta_k>1$,

$$ \widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k-1}{\alpha_k+\beta_k-2} $$

**Boundary cases:**

$$
\widehat{\theta}_{\mathrm{MAP}}^{(k)} =
\begin{cases}
\dfrac{\alpha_k-1}{\alpha_k+\beta_k-2}, &\alpha_k>1,\ \beta_k>1,\\[8pt]
0, &\alpha_k\leq1,\ \beta_k>1,\\[4pt]
1, &\alpha_k>1,\ \beta_k\leq1.
\end{cases}
$$

For $\alpha_k=\beta_k=1$, the density is uniform, so every point in $[0,1]$ is a mode. For plotting, we may use:

$$ \widehat{\theta}_{\mathrm{MAP}}^{(0)} = 0.5 $$

### Posterior variance

The posterior variance is:

$$ \operatorname{Var}(\Theta\mid\mathbf{y}^{(k)}) = \frac{\alpha_k\beta_k}{(\alpha_k+\beta_k)^2(\alpha_k+\beta_k+1)} $$

### 6. Performance Tracking and Convergence Analysis

In [4]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ---------------------------------------------------------
# Simulation settings
# ---------------------------------------------------------
rng = np.random.default_rng(42)

theta_true = 0.35
n_impressions = 100

alpha_0 = 1.0
beta_0 = 1.0

# ---------------------------------------------------------
# MAP function, including boundary cases
# ---------------------------------------------------------
def beta_map(alpha, beta_param):
    if alpha > 1 and beta_param > 1:
        return (alpha - 1) / (alpha + beta_param - 2)

    if alpha == 1 and beta_param == 1:
        return 0.5

    if alpha <= 1 and beta_param > 1:
        return 0.0

    if alpha > 1 and beta_param <= 1:
        return 1.0

    return np.nan

# ---------------------------------------------------------
# Initial state
# ---------------------------------------------------------
alpha_k = alpha_0
beta_k = beta_0

steps = [0]

posterior_means = [
    alpha_k / (alpha_k + beta_k)
]

map_estimates = [
    beta_map(alpha_k, beta_k)
]

posterior_variances = [
    (
        alpha_k * beta_k
        /
        (
            (alpha_k + beta_k)**2
            * (alpha_k + beta_k + 1)
        )
    )
]

records = []

# ---------------------------------------------------------
# Sequential Bayesian updating
# ---------------------------------------------------------
for k in range(1, n_impressions + 1):

    # Simulate Bernoulli response using U(0,1)
    uniform_draw = rng.uniform(0.0, 1.0)

    y_k = int(
        uniform_draw < theta_true
    )

    # Closed-form parameter updates
    alpha_k = alpha_k + y_k
    beta_k = beta_k + 1 - y_k

    # Posterior mean
    posterior_mean = (
        alpha_k /
        (alpha_k + beta_k)
    )

    # MAP estimate
    posterior_map = beta_map(
        alpha_k,
        beta_k
    )

    # Posterior variance
    posterior_variance = (
        alpha_k * beta_k
        /
        (
            (alpha_k + beta_k)**2
            * (alpha_k + beta_k + 1)
        )
    )

    steps.append(k)
    posterior_means.append(posterior_mean)
    map_estimates.append(posterior_map)
    posterior_variances.append(
        posterior_variance
    )

    records.append({
        "Step": k,
        "Response": y_k,
        "Alpha": alpha_k,
        "Beta": beta_k,
        "Posterior Mean": posterior_mean,
        "MAP": posterior_map,
        "Posterior Variance": posterior_variance
    })

# ---------------------------------------------------------
# Results table
# ---------------------------------------------------------
results = pd.DataFrame(records)

display(results.round(4))

print(f"Total clicks: {int(alpha_k - alpha_0)}")
print(f"Total non-clicks: {int(beta_k - beta_0)}")

print(
    f"Final posterior distribution: "
    f"Beta({alpha_k:.0f}, {beta_k:.0f})"
)

print(
    f"Final posterior mean: "
    f"{posterior_means[-1]:.4f}"
)

print(
    f"Final MAP estimate: "
    f"{map_estimates[-1]:.4f}"
)

print(
    f"Final posterior variance: "
    f"{posterior_variances[-1]:.6f}"
)

# ---------------------------------------------------------
# Plot estimator progression
# ---------------------------------------------------------
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines",
        name="MAP Estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True CTR = 0.35",
    annotation_position="top right"
)

fig.update_layout(
    title="Sequential Bayesian Estimation of Advertisement CTR",
    xaxis_title="Number of Impressions, k",
    yaxis_title="Estimated CTR",
    template="plotly_white",
    width=950,
    height=550
)

fig.update_xaxes(
    tickmode="linear",
    dtick=10
)

fig.update_yaxes(
    range=[0, 1]
)

fig.show()

,Step,Response,Alpha,Beta,Posterior Mean,MAP,Posterior Variance
0,1,0,1.0,2.0,0.3333,0.0000,0.0556
1,2,0,1.0,3.0,0.2500,0.0000,0.0375
2,3,0,1.0,4.0,0.2000,0.0000,0.0267
3,4,0,1.0,5.0,0.1667,0.0000,0.0198
4,5,1,2.0,5.0,0.2857,0.2000,0.0255
...,...,...,...,...,...,...,...
95,96,0,32.0,66.0,0.3265,0.3229,0.0022
96,97,0,32.0,67.0,0.3232,0.3196,0.0022
97,98,1,33.0,67.0,0.3300,0.3265,0.0022
98,99,1,34.0,67.0,0.3366,0.3333,0.0022


Total clicks: 33
Total non-clicks: 67
Final posterior distribution: Beta(34, 68)
Final posterior mean: 0.3333
Final MAP estimate: 0.3300
Final posterior variance: 0.002157


## Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

### 1.Prior Belief Boundaries

The initial belief about the remaining stiffness efficiency is $\Theta \sim \operatorname{Beta}(8, 1.5)$.

The Beta density is:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{B(8,1.5)} \theta^{8-1}(1-\theta)^{1.5-1}, \qquad 0<\theta<1$$

Therefore,

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{B(8,1.5)} \theta^7(1-\theta)^{0.5}$$

### Expected prior stiffness

For a Beta distribution,

$$\mathbb{E}[\Theta] = \frac{\alpha}{\alpha+\beta}$$

Hence,

$$\mathbb{E}[\Theta^{(0)}] = \frac{8}{8+1.5} = \frac{8}{9.5} \approx 0.8421$$

Thus, before receiving sensor measurements, engineers expect the structure to retain approximately **84.21%** of its nominal stiffness.

The distribution is suitable because it is restricted to the physically meaningful interval $(0,1)$ and places most probability near high stiffness values. It therefore represents an initially optimistic belief that a newly manufactured or recently inspected component is likely to be healthy, while still allowing some probability of degradation.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta as beta_dist
from scipy.integrate import trapezoid

# Beta prior parameters
alpha = 8.0
beta_param = 1.5

# Avoid theta = 0 because the later likelihood contains log(theta)
theta_grid = np.linspace(0.01, 1.0, 4000)

# Evaluate the Beta prior
prior = beta_dist.pdf(theta_grid, alpha, beta_param)

# Normalize numerically over the selected grid
prior = prior / trapezoid(prior, theta_grid)

# Analytical prior mean
prior_mean = alpha / (alpha + beta_param)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=prior,
        mode="lines",
        name="Beta(8, 1.5) prior"
    )
)

fig.add_vline(
    x=prior_mean,
    line_dash="dash",
    annotation_text=f"Prior mean = {prior_mean:.4f}",
    annotation_position="top left"
)

fig.update_layout(
    title="Initial Prior Distribution of Remaining Stiffness",
    xaxis_title="Remaining stiffness efficiency, θ",
    yaxis_title="Probability density",
    template="plotly_white",
    width=900,
    height=520
)

fig.show()

print(f"Analytical prior mean = {prior_mean:.4f}")

Analytical prior mean = 0.8421


### 2. Structural Likelihood Formulation

The measurement model is:

$$y_k = \theta K_{\text{nominal}}e^{\epsilon_k}, \qquad \epsilon_k\sim\mathcal N(0,\sigma^2)$$

Taking natural logarithms gives:

$$\ln y_k = \ln(\theta K_{\text{nominal}}) + \epsilon_k$$

Therefore,

$$\ln Y_k\mid\Theta=\theta \sim \mathcal N \left( \ln(\theta K_{\text{nominal}}), \sigma^2 \right)$$

Hence, $Y_k\mid\Theta=\theta$ has a log-normal distribution.

The likelihood contribution of a single measurement $y_k>0$ is:

$$L(y_k\mid\theta) = \frac{1}{y_k\sigma\sqrt{2\pi}} \exp\left[ -\frac{ \left( \ln y_k-\ln(\theta K_{\text{nominal}}) \right)^2 }{ 2\sigma^2 } \right]$$

for $0<\theta\leq1$.

An equivalent expression is:

$$L(y_k\mid\theta) = \frac{1}{y_k\sigma\sqrt{2\pi}} \exp\left[ -\frac{ \left( \ln\frac{y_k}{\theta K_{\text{nominal}}} \right)^2 }{ 2\sigma^2 } \right]$$

Assuming that sensor measurements are conditionally independent given $\Theta=\theta$, the joint likelihood for the running history $\mathbf y^{(k)} = (y_1,\ldots,y_k)$ is:

$$L(\mathbf y^{(k)}\mid\theta) = \prod_{i=1}^{k} \frac{1}{y_i\sigma\sqrt{2\pi}} \exp\left[ -\frac{ \left( \ln\frac{y_i}{\theta K_{\text{nominal}}} \right)^2 }{ 2\sigma^2 } \right]$$


### 3. Mathematical Formulation of the Non-Conjugate Grid Update

The Beta prior has the form:

$$f_{\Theta}^{(0)}(\theta) \propto \theta^{\alpha-1} (1-\theta)^{\beta-1}$$

The log-normal likelihood contains the term:

$$\exp\left[ -\frac{ \left( \ln\frac{y_k}{\theta K_{\text{nominal}}} \right)^2 }{ 2\sigma^2 } \right]$$

Multiplying these two expressions does not produce another Beta density. Therefore, the Beta prior is not conjugate to this log-normal likelihood, and the posterior cannot be written as a standard distribution with analytically updated parameters.

At step $k$, the previous posterior becomes the new prior. The recursive update is:

$$f_{\Theta\mid\mathbf{Y}^{(k)}} \left( \theta\mid\mathbf{y}^{(k)} \right) \propto L(y_k\mid\theta) f_{\Theta\mid\mathbf{Y}^{(k-1)}} \left( \theta\mid\mathbf{y}^{(k-1)} \right)$$

Using shorter notation:

$$f_k(\theta) \propto L(y_k\mid\theta)f_{k-1}(\theta)$$

The normalized posterior is:

$$f_k(\theta) = \frac{L(y_k\mid\theta)f_{k-1}(\theta)}{\displaystyle \int_0^1 L(y_k\mid s)f_{k-1}(s)\,ds}$$

The denominator ensures that:

$$\int_0^1 f_k(\theta)\,d\theta = 1$$

### 4. Running Point Estimates

The Bayesian estimator under squared-error loss is the posterior mean:

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb{E}[\Theta\mid\mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \int_0^1 \theta f_k(\theta)\,d\theta$$

### Running MAP estimate
The Maximum A Posteriori estimate is the value of $\theta$ at which the posterior density is greatest:

$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \operatorname*{arg\,max}_{0<\theta\leq1} f_k(\theta)$$

The posterior mean uses the entire distribution, whereas the MAP estimate uses only the location of its highest point.

### 5. Algorithmic Grid Approximation and Normalization

A fixed numerical grid is selected over the physical range:

$$0 < \theta \leq 1$$

Because the likelihood contains $\ln\theta$, the value $\theta=0$ must not be included. A computational range such as:

$$0.01 \leq \theta \leq 1.0$$

can therefore be used.

The numerical procedure is:

1. **Create a fine grid** $\theta_1, \theta_2, \ldots, \theta_M$.
2. **Evaluate the initial Beta prior** at every grid point.
3. **Normalize the initial grid density** using the trapezoidal rule.
4. **When measurement $y_k$ arrives, evaluate the likelihood** at every grid point:
   $$L_j = L(y_k\mid\theta_j)$$
5. **Multiply the previous posterior by the new likelihood:**
   $$q_j = L_j f_{k-1}(\theta_j)$$
6. **Approximate the normalizing constant:**
   $$Z_k \approx \int_{0.01}^{1} q(\theta)\,d\theta$$
   Using NumPy, ...

### 6. Performance Tracking and Degradation Convergence Analysis



In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from scipy.stats import beta as beta_dist
from scipy.integrate import trapezoid

# ---------------------------------------------------------
# Reproducible simulation settings
# ---------------------------------------------------------
rng = np.random.default_rng(42)

theta_true = 0.68
K_nominal = 50.0        # kN/mm
sigma = 0.15            # log-space noise standard deviation
n_measurements = 15

alpha = 8.0
beta_param = 1.5

# ---------------------------------------------------------
# Fixed bounded grid
# ---------------------------------------------------------
# theta = 0 is excluded because log(theta) is undefined.
theta_grid = np.linspace(0.01, 1.0, 5000)

# Initial Beta prior
posterior = beta_dist.pdf(
    theta_grid,
    alpha,
    beta_param
)

posterior = posterior / trapezoid(
    posterior,
    theta_grid
)

# ---------------------------------------------------------
# Helper function for posterior summaries
# ---------------------------------------------------------
def posterior_summary(theta_grid, density):
    mean = trapezoid(
        theta_grid * density,
        theta_grid
    )

    map_estimate = theta_grid[
        np.argmax(density)
    ]

    variance = trapezoid(
        (theta_grid - mean) ** 2 * density,
        theta_grid
    )

    # Numerical cumulative distribution for credible interval
    increments = (
        0.5
        * (density[:-1] + density[1:])
        * np.diff(theta_grid)
    )

    cdf = np.concatenate([
        [0.0],
        np.cumsum(increments)
    ])

    cdf = cdf / cdf[-1]

    lower_95 = np.interp(
        0.025,
        cdf,
        theta_grid
    )

    upper_95 = np.interp(
        0.975,
        cdf,
        theta_grid
    )

    return (
        mean,
        map_estimate,
        variance,
        lower_95,
        upper_95
    )

# ---------------------------------------------------------
# Step 0 values
# ---------------------------------------------------------
(
    initial_mean,
    initial_map,
    initial_variance,
    initial_lower,
    initial_upper
) = posterior_summary(theta_grid, posterior)

steps = [0]
posterior_means = [initial_mean]
map_estimates = [initial_map]
posterior_variances = [initial_variance]
lower_limits = [initial_lower]
upper_limits = [initial_upper]

records = []

# Store posterior curves at requested milestones
milestones = {0, 1, 2, 5, 10, 15}
stored_posteriors = {
    0: posterior.copy()
}

# ---------------------------------------------------------
# Sequential sensor updating
# ---------------------------------------------------------
for k in range(1, n_measurements + 1):

    # Simulate log-normal measurement noise
    epsilon_k = rng.normal(
        loc=0.0,
        scale=sigma
    )

    # Generate sensor measurement from the physical model
    y_k = (
        theta_true
        * K_nominal
        * np.exp(epsilon_k)
    )

    # Log-normal likelihood evaluated across theta grid
    log_ratio = np.log(
        y_k / (theta_grid * K_nominal)
    )

    likelihood = (
        1.0
        / (y_k * sigma * np.sqrt(2.0 * np.pi))
        * np.exp(
            -(log_ratio ** 2)
            / (2.0 * sigma ** 2)
        )
    )

    # Unnormalized recursive posterior
    unnormalized = likelihood * posterior

    # Sequential trapezoidal normalization
    normalizer = trapezoid(
        unnormalized,
        theta_grid
    )

    if (
        not np.isfinite(normalizer)
        or normalizer <= 0
    ):
        raise RuntimeError(
            f"Invalid normalizer at step {k}"
        )

    posterior = unnormalized / normalizer

    # Calculate running summaries
    (
        posterior_mean,
        posterior_map,
        posterior_variance,
        lower_95,
        upper_95
    ) = posterior_summary(
        theta_grid,
        posterior
    )

    steps.append(k)
    posterior_means.append(posterior_mean)
    map_estimates.append(posterior_map)
    posterior_variances.append(
        posterior_variance
    )
    lower_limits.append(lower_95)
    upper_limits.append(upper_95)

    records.append({
        "Step": k,
        "Sensor measurement": y_k,
        "Posterior mean": posterior_mean,
        "MAP estimate": posterior_map,
        "Posterior SD": np.sqrt(
            posterior_variance
        ),
        "95% lower": lower_95,
        "95% upper": upper_95
    })

    if k in milestones:
        stored_posteriors[k] = posterior.copy()

# ---------------------------------------------------------
# Display numerical results
# ---------------------------------------------------------
results = pd.DataFrame(records)
display(results.round(4))

print(f"Prior mean: {initial_mean:.4f}")
print(f"Prior MAP: {initial_map:.4f}")

print(
    f"Final posterior mean: "
    f"{posterior_means[-1]:.4f}"
)

print(
    f"Final MAP estimate: "
    f"{map_estimates[-1]:.4f}"
)

print(
    f"Final posterior SD: "
    f"{np.sqrt(posterior_variances[-1]):.4f}"
)

print(
    f"Final 95% credible interval: "
    f"[{lower_limits[-1]:.4f}, "
    f"{upper_limits[-1]:.4f}]"
)

# ---------------------------------------------------------
# Optional explicit confidence criterion
# ---------------------------------------------------------
# "Confident isolation" is defined here as:
# 1. Both estimates are within 0.02 of theta_true
# 2. Posterior standard deviation is below 0.05
confident_step = None

for k in range(1, n_measurements + 1):
    mean_close = (
        abs(posterior_means[k] - theta_true)
        < 0.02
    )

    map_close = (
        abs(map_estimates[k] - theta_true)
        < 0.02
    )

    sufficiently_narrow = (
        np.sqrt(posterior_variances[k])
        < 0.05
    )

    if mean_close and map_close and sufficiently_narrow:
        confident_step = k
        break

print(
    "First step satisfying the stated confidence "
    f"criterion: {confident_step}"
)

# ---------------------------------------------------------
# Plot 1: Full posterior curves at milestones
# ---------------------------------------------------------
fig1 = go.Figure()

for k in sorted(milestones):
    fig1.add_trace(
        go.Scatter(
            x=theta_grid,
            y=stored_posteriors[k],
            mode="lines",
            name=f"k = {k}"
        )
    )

fig1.add_vline(
    x=theta_true,
    line_dash="dash",
    annotation_text="True stiffness = 0.68",
    annotation_position="top left"
)

fig1.update_layout(
    title="Sequential Posterior Density of Remaining Stiffness",
    xaxis_title="Remaining stiffness efficiency, θ",
    yaxis_title="Posterior density",
    template="plotly_white",
    width=950,
    height=570
)

fig1.show()

# ---------------------------------------------------------
# Plot 2: Posterior mean and MAP timeline
# ---------------------------------------------------------
fig2 = go.Figure()

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True stiffness = 0.68",
    annotation_position="top left"
)

fig2.update_layout(
    title="Convergence of Sequential Stiffness Estimates",
    xaxis_title="Number of sensor readings, k",
    yaxis_title="Estimated stiffness efficiency",
    template="plotly_white",
    width=950,
    height=550
)

fig2.update_xaxes(
    tickmode="linear",
    dtick=1
)

fig2.show()

,Step,Sensor measurement,Posterior mean,MAP estimate,Posterior SD,95% lower,95% upper
0,1,35.5901,0.7947,0.7972,0.0926,0.6107,0.9629
1,2,29.0891,0.6977,0.6877,0.0719,0.5661,0.8476
2,3,38.0510,0.7177,0.7107,0.0609,0.6051,0.8436
3,4,39.1518,0.7332,0.7277,0.0541,0.6325,0.8443
4,5,25.3735,0.6823,0.6780,0.0454,0.5974,0.7754
5,6,27.9672,0.6604,0.6568,0.0402,0.5849,0.7426
6,7,34.6583,0.6649,0.6617,0.0375,0.5943,0.7414
7,8,32.4248,0.6629,0.6602,0.0350,0.5968,0.7341
8,9,33.9144,0.6645,0.6621,0.0331,0.6020,0.7318
9,10,29.9163,0.6577,0.6554,0.0311,0.5988,0.7207


Prior mean: 0.8421
Prior MAP: 0.9333
Final posterior mean: 0.6873
Final MAP estimate: 0.6857
Final posterior SD: 0.0266
Final 95% credible interval: [0.6367, 0.7408]
First step satisfying the stated confidence criterion: 5


### Analysis

The initial Beta prior has mean $\mathbb{E}[\Theta^{(0)}] = 0.8421$ and a mode near $0.9333$. Therefore, the system initially strongly favors a healthy or only slightly degraded structure.

Using the fixed random seed 42, the posterior estimates evolve approximately as follows:

| Step | Posterior mean | MAP | 95% credible interval |
|---|---|---|---|
| 0 | 0.8421 | 0.9333 | [0.5668, 0.9870] |
| 1 | 0.7947 | 0.7972 | [0.6107, 0.9629] |
| 2 | 0.6977 | 0.6877 | [0.5661, 0.8476] |
| 5 | 0.6823 | 0.6780 | [0.5974, 0.7754] |
| 10 | 0.6577 | 0.6554 | [0.5988, 0.7207] |
| 15 | 0.6873 | 0.6857 | [0.6367, 0.7408] |

The first two measurements strongly reduce the initially optimistic estimate. By approximately the fifth sensor reading, both the posterior mean and MAP are very close to the true stiffness value $\theta_{\text{true}}=0.68$.

Using the explicit criterion that both estimates must be within 0.02 of the true value and the posterior standard deviation must be below 0.05, the first confident identification occurs at approximately $k=5$.

However, individual noisy measurements may temporarily move the estimates away from the true value. Consequently, the result should be described as approximately five readings to overcome the optimistic prior, while ten to fifteen readings produce a more consistently narrow posterior.

At step 15,

$$\widehat{\theta}_{\mathrm{Bayes}}^{(15)} \approx 0.6873$$

and

$$\widehat{\theta}_{\mathrm{MAP}}^{(15)} \approx 0.6857$$

The final 95% credible interval is approximately $[0.6367, 0.7408]$.

The narrowing of the posterior density means that the monitoring system is becoming more certain that the structure has suffered a substantial stiffness reduction. The distribution is no longer spread broadly across healthy and damaged states; it becomes concentrated near the 68% stiffness state.

If engineers define a safety threshold $\theta_{\text{safe}}$, the posterior probability of unsafe operation can be calculated as:

$$P(\Theta<\theta_{\text{safe}}\mid\mathbf y^{(k)}) = \int_0^{\theta_{\text{safe}}} f_k(\theta)\,d\theta$$

A narrow posterior lying mainly below the safety threshold provides strong evidence for inspection, maintenance, load restriction, or component replacement. The posterior width is therefore important because it distinguishes an uncertain warning from a confident detection of structural degradation.

## Q. Gaussian Mixture Clustering as Conditional Updating

### 1. Deriving the Marginal Density

Using the law of total probability,

$$p(x_i) = \sum_{k=1}^{K} p(x_i,C_i=k)$$

Using $p(x_i,C_i=k) = p(x_i\mid C_i=k)P(C_i=k)$, we obtain:

$$p(x_i) = \sum_{k=1}^{K} p(x_i\mid C_i=k)P(C_i=k)$$

Since $P(C_i=k)=\phi_k$ and $X_i\mid C_i=k \sim \mathcal{N}(\mu_k,\Sigma_k)$, then $p(x_i\mid C_i=k) = \mathcal{N}(x_i\mid\mu_k,\Sigma_k)$.

Therefore,

$$p(x_i) = \sum_{k=1}^{K} \phi_k \mathcal{N}(x_i\mid\mu_k,\Sigma_k)$$

where $\phi_k\geq 0$ and $\sum_{k=1}^{K}\phi_k=1$.

It is called a Gaussian mixture density because it is a weighted sum of $K$ Gaussian component densities.

$$\text{Mixture density} = \sum \text{mixture weight} \times \text{Gaussian density}$$

Unlike a single Gaussian distribution, a GMM can represent:
* Multiple peaks
* Asymmetry
* Different covariance structures

### 2. Deriving the Posterior Cluster Probability

For a fixed observation $x_i$, Bayes' rule gives:

$$P(C_i=k\mid X_i=x_i) = \frac{p(x_i\mid C_i=k)P(C_i=k)}{p(x_i)}$$

The marginal density is:

$$p(x_i) = \sum_{j=1}^{K} p(x_i\mid C_i=j)P(C_i=j)$$

Therefore,

$$P(C_i=k\mid X_i=x_i) = \frac{p(x_i\mid C_i=k)P(C_i=k)}{\sum_{j=1}^{K} p(x_i\mid C_i=j)P(C_i=j)}$$

Substitute $P(C_i=k)=\phi_k$ and $p(x_i\mid C_i=k) = \mathcal{N}(x_i\mid\mu_k,\Sigma_k)$. Then,

$$\gamma_{ik} = P(C_i=k\mid X_i=x_i) = \frac{\phi_k\mathcal{N}(x_i\mid\mu_k,\Sigma_k)}{\sum_{j=1}^{K} \phi_j\mathcal{N}(x_i\mid\mu_j,\Sigma_j)}$$

The numerator is:

$$\text{prior cluster probability} \times \text{compatibility with that cluster}$$

and the denominator normalizes over all clusters. Hence, $0 \leq \gamma_{ik} \leq 1$ and:

$$\sum_{k=1}^{K}\gamma_{ik}=1$$

Thus, $\gamma_{ik}$ is the posterior probability that observation $x_i$ belongs to cluster $k$.

### 3. One-Hot Encoding of the Latent Cluster Variable

Define

$$Z_{ik} = \begin{cases} 1, & C_i=k, \\ 0, & C_i\neq k. \end{cases}$$

Since $Z_{ik}$ is an indicator variable,

$$
\begin{aligned}
\mathbb{E}[Z_{ik}\mid X_i=x_i] &= 1\cdot P(Z_{ik}=1\mid X_i=x_i) \\
&\quad+ 0\cdot P(Z_{ik}=0\mid X_i=x_i).
\end{aligned}
$$

Therefore,

$$\mathbb{E}[Z_{ik}\mid X_i=x_i] = P(Z_{ik}=1\mid X_i=x_i)$$

Because $Z_{ik}=1 \iff C_i=k$, we obtain:

$$\mathbb{E}[Z_{ik}\mid X_i=x_i] = P(C_i=k\mid X_i=x_i) = \gamma_{ik}$$

For the complete one-hot vector,

$$
Z_i = \begin{bmatrix} Z_{i1} \\ Z_{i2} \\ \vdots \\ Z_{iK} \end{bmatrix}
$$

the conditional expectation is:

$$
\mathbb{E}[Z_i\mid X_i=x_i] = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}
$$

Therefore,

$$\text{soft cluster assignment} = \mathbb{E}[Z_i\mid X_i=x_i]$$

For example,

$$
\mathbb{E}[Z_i\mid X_i=x_i] = \begin{bmatrix} 0.55 \\ 0.40 \\ 0.05 \end{bmatrix}
$$

means that $x_i$ has posterior membership probabilities 55%, 40%, and 5% for the three clusters.

### 4. From Soft Assignment to Hard Clustering

The soft assignment is:

$$ \boldsymbol{\gamma}_i = (\gamma_{i1},\ldots,\gamma_{iK})^T $$

It preserves all cluster membership probabilities.

The hard assignment is:

$$ \widehat{C}_i = \operatorname*{arg\,max}_{1\leq k\leq K} \gamma_{ik} $$

For example, suppose:

$$ \boldsymbol{\gamma}_i = (0.48,0.49,0.03)^T $$

Then

$$ \widehat{C}_i = 2 $$

**Soft clustering**
$(0.48,0.49,0.03)$ shows that clusters 1 and 2 are almost equally likely.

**Hard clustering**
$\widehat{C}_i=2$ keeps only the winning cluster.

Thus,

**Hard assignment discards posterior uncertainty and non-winning probabilities.**



### 5. Conditional Expectation of the Observation Given the Cluster

The model assumes:

$$X_i\mid C_i=k \sim \mathcal{N}(\mu_k,\Sigma_k)$$

The expectation of a multivariate Gaussian is its mean vector. Therefore,

$$\mathbb{E}[X_i\mid C_i=k] = \mu_k$$

Hence, $\mu_k$ represents the center of cluster $k$.

The two conditional expectations have different meanings:

$$
\mathbb{E}[Z_i\mid X_i=x_i] = \begin{bmatrix} \gamma_{i1}\\ \vdots\\ \gamma_{iK} \end{bmatrix}
$$

gives the posterior cluster membership probabilities for one observed point.

In contrast,

$$ \mathbb{E}[X_i\mid C_i=k] = \mu_k $$

gives the average spatial location of cluster $k$.

Therefore,

$$
\begin{aligned}
\mathbb{E}[Z_i\mid X_i=x_i] &= \text{soft membership of point } x_i, \\
\mathbb{E}[X_i\mid C_i=k] &= \text{center of cluster } k.
\end{aligned}
$$

### 6. Complete-Data Likelihood

When the latent indicators $z_{ik}$ are known,

$$p(x_1,\ldots,x_n,z_1,\ldots,z_n) = \prod_{i=1}^{n} \prod_{k=1}^{K} \left[ \phi_k \mathcal{N}(x_i\mid\mu_k,\Sigma_k) \right]^{z_{ik}}$$

Take the logarithm:

$$\ell_c = \log \prod_{i=1}^{n} \prod_{k=1}^{K} \left[ \phi_k \mathcal{N}(x_i\mid\mu_k,\Sigma_k) \right]^{z_{ik}}$$

Using $\log\prod_r a_r = \sum_r \log a_r$, we obtain:

$$\ell_c = \sum_{i=1}^{n} \sum_{k=1}^{K} z_{ik} \log \left[ \phi_k \mathcal{N}(x_i\mid\mu_k,\Sigma_k) \right]$$

Using $\log(ab) = \log a + \log b$, the result is:

$$\ell_c = \sum_{i=1}^{n} \sum_{k=1}^{K} z_{ik} \left[ \log\phi_k + \log\mathcal{N}(x_i\mid\mu_k,\Sigma_k) \right]$$

If $z_{ik}$ were known, every observation would already have a known cluster. The parameters could then be estimated separately for each cluster:

$$ \phi_k = \frac{\text{number of observations in cluster } k}{n} $$

$$ \mu_k = \text{mean of observations in cluster } k $$

$$ \Sigma_k = \text{covariance of observations in cluster } k $$

The main difficulty is that $z_{ik}$ is hidden.

### 7. EM Interpretation

Let the current parameter estimates at iteration $t$ be:

$$ \phi_k^{(t)}, \qquad \mu_k^{(t)}, \qquad \Sigma_k^{(t)} $$

#### E-step
The responsibilities are calculated as:

$$ \gamma_{ik}^{(t)} = \frac{\phi_k^{(t)} \mathcal{N}\left(x_i\mid \mu_k^{(t)},\Sigma_k^{(t)}\right)}{\displaystyle\sum_{j=1}^{K} \phi_j^{(t)} \mathcal{N}\left(x_i\mid \mu_j^{(t)},\Sigma_j^{(t)}\right)} $$

The unknown indicator is replaced by its conditional expectation:

$$ z_{ik} \leadsto \mathbb{E}[Z_{ik}\mid X_i=x_i] = \gamma_{ik}^{(t)} $$

The expected complete-data log-likelihood is:

$$ Q = \sum_{i=1}^{n} \sum_{k=1}^{K} \gamma_{ik}^{(t)} \left[ \log\phi_k + \log\mathcal{N}(x_i\mid\mu_k,\Sigma_k) \right] $$

The E-step is a conditional update because:

$$ \text{prior membership probability} = \phi_k $$

is updated using the observed point $x_i$ to produce:

$$ \text{posterior membership probability} = \gamma_{ik} $$

Thus,

$$ \phi_k \xrightarrow{\text{observe } x_i} \gamma_{ik} $$

### 8. Parameter Updates

Define the effective cluster size:

$$ N_k = \sum_{i=1}^{n}\gamma_{ik} $$

Unlike an ordinary count, $N_k$ may be non-integer because each observation contributes fractionally.

### 8.1 Mixture-weight update

The relevant part of $Q$ is:

$$ Q_\phi = \sum_{k=1}^{K} N_k\log\phi_k $$

Subject to:

$$ \sum_{k=1}^{K}\phi_k=1 $$

Construct the Lagrangian:

$$ \mathcal{L} = \sum_{k=1}^{K}N_k\log\phi_k + \lambda \left( \sum_{k=1}^{K}\phi_k-1 \right) $$

Differentiate:

$$ \frac{\partial\mathcal{L}}{\partial\phi_k} = \frac{N_k}{\phi_k}+\lambda=0 $$

Hence,

$$ \phi_k = -\frac{N_k}{\lambda} $$

Using:

$$ \sum_{k=1}^{K}N_k=n $$

normalization gives:

$$ \lambda=-n $$

Therefore,

$$ \phi_k^{\mathrm{new}} = \frac{N_k}{n} $$

### 8.2 Mean update

The Gaussian log-density contains:

$$ -\frac{1}{2} (x_i-\mu_k)^T \Sigma_k^{-1} (x_i-\mu_k) $$

Differentiating $Q$ with respect to $\mu_k$:

$$ \frac{\partial Q}{\partial\mu_k} = \Sigma_k^{-1} \sum_{i=1}^{n} \gamma_{ik}(x_i-\mu_k) $$

Set it equal to zero:

$$ \sum_{i=1}^{n} \gamma_{ik}x_i - \mu_k \sum_{i=1}^{n}\gamma_{ik} = 0 $$

Therefore,

$$ \mu_k^{\mathrm{new}} = \frac{1}{N_k} \sum_{i=1}^{n} \gamma_{ik}x_i $$

### 8.3 Covariance update

Maximizing $Q$ with respect to $\Sigma_k$ gives:

$$ \Sigma_k^{\mathrm{new}} = \frac{1}{N_k} \sum_{i=1}^{n} \gamma_{ik} (x_i-\mu_k^{\mathrm{new}}) (x_i-\mu_k^{\mathrm{new}})^T $$

Thus, the complete M-step is:

$$ N_k = \sum_{i=1}^{n}\gamma_{ik} $$

$$ \phi_k^{\mathrm{new}} = \frac{N_k}{n} $$

$$ \mu_k^{\mathrm{new}} = \frac{\sum_i\gamma_{ik}x_i}{N_k} $$

$$ \Sigma_k^{\mathrm{new}} = \frac{ \sum_i \gamma_{ik} (x_i-\mu_k^{\mathrm{new}}) (x_i-\mu_k^{\mathrm{new}})^T }{ N_k } $$

The value $\gamma_{ik}$ acts as a fractional membership weight. For example, $\gamma_{ik}=0.80$ means that observation $x_i$ contributes 80% of one observation to cluster $k$.

### 9. Interpretation

Gaussian mixture clustering repeatedly alternates between conditional probability updating and parameter estimation. The mixture weight:

$$ \phi_k = P(C_i=k) $$

is the prior probability of cluster $k$. The Gaussian density:

$$ \mathcal{N}(x_i\mid\mu_k,\Sigma_k) $$

measures how compatible observation $x_i$ is with that cluster. Bayes' rule combines these values to produce:

$$ \gamma_{ik} = P(C_i=k\mid X_i=x_i) $$

which is the posterior membership probability. Therefore, the soft assignment vector is:

$$ \mathbb{E}[Z_i\mid X_i=x_i] = (\gamma_{i1},\ldots,\gamma_{iK})^T $$

The M-step then updates $\phi_k$, $\mu_k$, and $\Sigma_k$ using these posterior probabilities as fractional weights. The process is repeated until the model parameters stabilize.

$$ \text{GMM clustering} = \text{probabilistic clustering using conditional expectations of latent memberships} $$

### 10. Computational Simulation and Out-of-Sample Validation

In [5]:
# Install required packages
!pip -q install kagglehub datawash-inspector scikit-learn plotly

In [6]:
from pathlib import Path
import os
import pandas as pd
import kagglehub

# ---------------------------------------------------------
# Download the Kaggle dataset
# ---------------------------------------------------------
dataset_directory = kagglehub.dataset_download(
    "arjunbhasin2013/ccdata"
)

csv_files = list(
    Path(dataset_directory).rglob("CC GENERAL.csv")
)

if not csv_files:
    raise FileNotFoundError(
        "CC GENERAL.csv was not found in the downloaded dataset."
    )

raw_csv_path = csv_files[0]

print("Dataset path:", raw_csv_path)

Using Colab cache for faster access to the 'ccdata' dataset.
Dataset path: /kaggle/input/ccdata/CC GENERAL.csv


In [7]:
from datawash import DataPipeline

cleaned_csv_path = "/content/cc_general_cleaned.csv"

pipe = DataPipeline()

pipe.load_data(str(raw_csv_path))

pipe.sanitize_garbage().auto_type_correct()

pipe.handle_missing_values(strategy="auto")

pipe.save_data(cleaned_csv_path)

# Optional interactive EDA dashboard
# pipe.show_dashboard()

df = pd.read_csv(cleaned_csv_path)

print("Cleaned dataset shape:", df.shape)
display(df.head())


Processing "/kaggle/input/ccdata/CC GENERAL.csv"...
INFO:datawash:
Processing "/kaggle/input/ccdata/CC GENERAL.csv"...
Success! Loaded dataset with 8950 rows and 18 columns.
INFO:datawash:Success! Loaded dataset with 8950 rows and 18 columns.
Sanitization complete: Garbage strings and pure whitespace converted to NaNs.
INFO:datawash:Sanitization complete: Garbage strings and pure whitespace converted to NaNs.
No columns required type conversion.
INFO:datawash:No columns required type conversion.

Imputation Strategy: 'auto'
INFO:datawash:
Imputation Strategy: 'auto'
  Successfully imputed: CREDIT_LIMIT, MINIMUM_PAYMENTS
INFO:datawash:  Successfully imputed: CREDIT_LIMIT, MINIMUM_PAYMENTS
Data saved to '/content/cc_general_cleaned.csv' (csv).
INFO:datawash:Data saved to '/content/cc_general_cleaned.csv' (csv).


Cleaned dataset shape: (8950, 18)


,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,864.206542,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


In [8]:
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


class GMMFinancialSegmenter:
    """
    Two-dimensional Gaussian Mixture Model for financial segmentation.
    """

    def __init__(
        self,
        features=("PURCHASES", "CREDIT_LIMIT"),
        n_components=3,
        test_size=0.20,
        random_state=42,
        covariance_type="full"
    ):
        self.features = tuple(features)
        self.n_components = n_components
        self.test_size = test_size
        self.random_state = random_state
        self.covariance_type = covariance_type

        self.scaler_ = None
        self.model_ = None
        self.is_fitted_ = False

    # -----------------------------------------------------
    # Prepare data and fit model
    # -----------------------------------------------------
    def fit(self, data):
        if isinstance(data, (str, Path)):
            dataframe = pd.read_csv(data)
        elif isinstance(data, pd.DataFrame):
            dataframe = data.copy()
        else:
            raise TypeError(
                "data must be a pandas DataFrame or CSV path."
            )

        missing_columns = [
            feature
            for feature in self.features
            if feature not in dataframe.columns
        ]

        if missing_columns:
            raise ValueError(
                f"Missing required columns: {missing_columns}"
            )

        # Keep the selected continuous features
        X = dataframe.loc[:, self.features].copy()

        # Ensure numerical values
        for feature in self.features:
            X[feature] = pd.to_numeric(
                X[feature],
                errors="coerce"
            )

        X = X.replace(
            [np.inf, -np.inf],
            np.nan
        ).dropna()

        if len(X) < self.n_components:
            raise ValueError(
                "Not enough valid observations for the GMM."
            )

        self.data_raw_ = X.reset_index(drop=True)

        # 80% training and 20% testing
        (
            self.train_raw_,
            self.test_raw_
        ) = train_test_split(
            self.data_raw_,
            test_size=self.test_size,
            random_state=self.random_state,
            shuffle=True
        )

        self.train_raw_ = self.train_raw_.reset_index(
            drop=True
        )

        self.test_raw_ = self.test_raw_.reset_index(
            drop=True
        )

        # Fit scaler using training data only
        self.scaler_ = StandardScaler()

        self.train_scaled_ = self.scaler_.fit_transform(
            self.train_raw_
        )

        self.test_scaled_ = self.scaler_.transform(
            self.test_raw_
        )

        # Fit the Gaussian mixture using EM
        self.model_ = GaussianMixture(
            n_components=self.n_components,
            covariance_type=self.covariance_type,
            n_init=10,
            max_iter=500,
            tol=1e-4,
            reg_covar=1e-6,
            random_state=self.random_state
        )

        self.model_.fit(self.train_scaled_)

        # Responsibilities
        self.train_responsibilities_ = (
            self.model_.predict_proba(
                self.train_scaled_
            )
        )

        self.test_responsibilities_ = (
            self.model_.predict_proba(
                self.test_scaled_
            )
        )

        # Hard assignments
        self.train_labels_ = np.argmax(
            self.train_responsibilities_,
            axis=1
        )

        self.test_labels_ = np.argmax(
            self.test_responsibilities_,
            axis=1
        )

        # Average log-likelihoods
        self.train_log_likelihood_ = self.model_.score(
            self.train_scaled_
        )

        self.test_log_likelihood_ = self.model_.score(
            self.test_scaled_
        )

        # Component centers converted to original units
        self.cluster_centers_raw_ = (
            self.scaler_.inverse_transform(
                self.model_.means_
            )
        )

        self.is_fitted_ = True

        print("GMM fitting completed")
        print("---------------------")
        print(
            "Converged:",
            self.model_.converged_
        )
        print(
            "EM iterations:",
            self.model_.n_iter_
        )
        print(
            "Training observations:",
            len(self.train_raw_)
        )
        print(
            "Test observations:",
            len(self.test_raw_)
        )
        print(
            "Average training log-likelihood:",
            round(self.train_log_likelihood_, 5)
        )
        print(
            "Average test log-likelihood:",
            round(self.test_log_likelihood_, 5)
        )

        return self

    # -----------------------------------------------------
    # Ensure model has been fitted
    # -----------------------------------------------------
    def _check_fitted(self):
        if not self.is_fitted_:
            raise RuntimeError(
                "Call fit() before using this method."
            )

    # -----------------------------------------------------
    # Empirical 2D density heatmap
    # -----------------------------------------------------
    def plot_empirical_density(self):
        self._check_fitted()

        feature_x, feature_y = self.features

        fig = px.density_heatmap(
            self.train_raw_,
            x=feature_x,
            y=feature_y,
            nbinsx=60,
            nbinsy=60,
            marginal_x="histogram",
            marginal_y="histogram",
            title=(
                "Empirical 2D Density of Training Data"
            )
        )

        fig.update_layout(
            template="plotly_white",
            width=950,
            height=700
        )

        return fig

    # -----------------------------------------------------
    # Create a responsibility surface
    # -----------------------------------------------------
    def _responsibility_surface(
        self,
        grid_size=250,
        quantile_window=(0.005, 0.995)
    ):
        self._check_fitted()

        feature_x, feature_y = self.features

        if quantile_window is None:
            x_min = self.data_raw_[feature_x].min()
            x_max = self.data_raw_[feature_x].max()

            y_min = self.data_raw_[feature_y].min()
            y_max = self.data_raw_[feature_y].max()
        else:
            lower, upper = quantile_window

            x_min, x_max = self.data_raw_[
                feature_x
            ].quantile([lower, upper])

            y_min, y_max = self.data_raw_[
                feature_y
            ].quantile([lower, upper])

        x_padding = 0.05 * max(x_max - x_min, 1.0)
        y_padding = 0.05 * max(y_max - y_min, 1.0)

        x_min -= x_padding
        x_max += x_padding
        y_min -= y_padding
        y_max += y_padding

        x_values = np.linspace(
            x_min,
            x_max,
            grid_size
        )

        y_values = np.linspace(
            y_min,
            y_max,
            grid_size
        )

        xx, yy = np.meshgrid(
            x_values,
            y_values
        )

        grid_raw = pd.DataFrame({
            feature_x: xx.ravel(),
            feature_y: yy.ravel()
        })

        grid_scaled = self.scaler_.transform(
            grid_raw
        )

        grid_responsibilities = (
            self.model_.predict_proba(
                grid_scaled
            )
        )

        maximum_responsibility = np.max(
            grid_responsibilities,
            axis=1
        ).reshape(xx.shape)

        hard_grid_label = np.argmax(
            grid_responsibilities,
            axis=1
        ).reshape(xx.shape)

        return {
            "x_values": x_values,
            "y_values": y_values,
            "max_responsibility":
                maximum_responsibility,
            "hard_grid_label": hard_grid_label,
            "bounds": (
                x_min,
                x_max,
                y_min,
                y_max
            )
        }

    # -----------------------------------------------------
    # General assignment plot
    # -----------------------------------------------------
    def _plot_assignments(
        self,
        data_raw,
        labels,
        responsibilities,
        title,
        grid_size=250
    ):
        surface = self._responsibility_surface(
            grid_size=grid_size
        )

        feature_x, feature_y = self.features

        (
            x_min,
            x_max,
            y_min,
            y_max
        ) = surface["bounds"]

        # Keep displayed points within the contour window
        display_mask = (
            data_raw[feature_x].between(
                x_min,
                x_max
            )
            &
            data_raw[feature_y].between(
                y_min,
                y_max
            )
        )

        shown_data = data_raw.loc[
            display_mask
        ].copy()

        shown_labels = labels[
            display_mask.to_numpy()
        ]

        shown_responsibilities = responsibilities[
            display_mask.to_numpy()
        ]

        maximum_point_responsibility = np.max(
            shown_responsibilities,
            axis=1
        )

        custom_data = np.column_stack([
            shown_labels,
            maximum_point_responsibility
        ])

        fig = go.Figure()

        # Continuous maximum-responsibility surface
        fig.add_trace(
            go.Contour(
                x=surface["x_values"],
                y=surface["y_values"],
                z=surface["max_responsibility"],
                colorscale="Viridis",
                opacity=0.75,
                contours=dict(
                    start=1.0 / self.n_components,
                    end=1.0,
                    size=0.05,
                    showlabels=True
                ),
                colorbar=dict(
                    title="Maximum<br>responsibility"
                ),
                name="Posterior confidence"
            )
        )

        # Observations with hard cluster colours
        fig.add_trace(
            go.Scattergl(
                x=shown_data[feature_x],
                y=shown_data[feature_y],
                mode="markers",
                marker=dict(
                    size=5,
                    color=shown_labels,
                    colorscale="Turbo",
                    opacity=0.70,
                    showscale=False
                ),
                customdata=custom_data,
                hovertemplate=(
                    f"{feature_x}: %{{x:.2f}}<br>"
                    f"{feature_y}: %{{y:.2f}}<br>"
                    "Hard cluster: %{customdata[0]:.0f}<br>"
                    "Maximum responsibility: "
                    "%{customdata[1]:.3f}"
                    "<extra></extra>"
                ),
                name="Observations"
            )
        )

        # Gaussian component means
        fig.add_trace(
            go.Scatter(
                x=self.cluster_centers_raw_[:, 0],
                y=self.cluster_centers_raw_[:, 1],
                mode="markers+text",
                marker=dict(
                    symbol="x",
                    size=16,
                    color="black",
                    line=dict(width=2)
                ),
                text=[
                    f"μ{k}"
                    for k in range(
                        self.n_components
                    )
                ],
                textposition="top center",
                name="Cluster means"
            )
        )

        fig.update_layout(
            title=title,
            xaxis_title=feature_x,
            yaxis_title=feature_y,
            template="plotly_white",
            width=1000,
            height=650
        )

        fig.update_xaxes(
            range=[x_min, x_max]
        )

        fig.update_yaxes(
            range=[y_min, y_max]
        )

        return fig

    # -----------------------------------------------------
    # Training assignment plot
    # -----------------------------------------------------
    def plot_training_assignments(
        self,
        grid_size=250
    ):
        self._check_fitted()

        return self._plot_assignments(
            data_raw=self.train_raw_,
            labels=self.train_labels_,
            responsibilities=(
                self.train_responsibilities_
            ),
            title=(
                "Training Data: GMM Responsibilities "
                "and Hard Assignments"
            ),
            grid_size=grid_size
        )

    # -----------------------------------------------------
    # Test assignment plot
    # -----------------------------------------------------
    def plot_test_assignments(
        self,
        grid_size=250
    ):
        self._check_fitted()

        return self._plot_assignments(
            data_raw=self.test_raw_,
            labels=self.test_labels_,
            responsibilities=(
                self.test_responsibilities_
            ),
            title=(
                "Out-of-Sample Test Data: "
                "GMM Responsibilities"
            ),
            grid_size=grid_size
        )

    # -----------------------------------------------------
    # Cluster summary
    # -----------------------------------------------------
    def cluster_summary(self):
        self._check_fitted()

        summary_data = self.train_raw_.copy()

        summary_data["Cluster"] = (
            self.train_labels_
        )

        summary_data["Maximum Responsibility"] = (
            np.max(
                self.train_responsibilities_,
                axis=1
            )
        )

        summary = summary_data.groupby(
            "Cluster"
        ).agg(
            Count=(
                self.features[0],
                "size"
            ),
            Mean_PURCHASES=(
                self.features[0],
                "mean"
            ),
            Mean_CREDIT_LIMIT=(
                self.features[1],
                "mean"
            ),
            Mean_Max_Responsibility=(
                "Maximum Responsibility",
                "mean"
            )
        )

        return summary

In [9]:
segmenter = GMMFinancialSegmenter(
    features=(
        "PURCHASES",
        "CREDIT_LIMIT"
    ),
    n_components=3,
    test_size=0.20,
    random_state=42,
    covariance_type="full"
)

segmenter.fit(df)

GMM fitting completed
---------------------
Converged: True
EM iterations: 31
Training observations: 7160
Test observations: 1790
Average training log-likelihood: -1.58979
Average test log-likelihood: -1.59019


In [10]:
display(
    segmenter.cluster_summary().round(3)
)

,Count,Mean_PURCHASES,Mean_CREDIT_LIMIT,Mean_Max_Responsibility
Cluster,,,,
0,3194,177.173,2043.627,0.916
1,689,5176.147,9816.981,0.923
2,3277,907.336,5842.654,0.914


In [11]:
fig_density = (
    segmenter.plot_empirical_density()
)

fig_density.show()

In [12]:
fig_training = (
    segmenter.plot_training_assignments(
        grid_size=250
    )
)

fig_training.show()

In [13]:
fig_test = (
    segmenter.plot_test_assignments(
        grid_size=250
    )
)

fig_test.show()